# Source-Level Group Statistics

This notebook performs **group-level source space statistics** across subjects.
It assumes that individual source analysis has already been completed for each subject
using `example_usage_source_complete.ipynb` (through Step 4: Visualize Statistical Results),
which produces morphed source estimates on the `fsaverage` template brain.

**Pipeline:**
1. Load and merge morphed source estimates across subjects
2. Visual check of group-average contrast
3. Cluster-based permutation testing (spatio-temporal)
4. Visualize significant clusters on the cortical surface
5. *(Optional)* Generate figure for publication

**Prerequisites:** morphed STC files saved as  
`S{i}_from_{freq_min}_to_{freq_max}_{orientation}_morph_surf_{condition}_stc`  
and the fsaverage source space `Av-{spacing}-src.fif` — both generated by
`example_usage_source_complete.ipynb`.

**Author:** Nikita Otstavnov, 2023  
**Refactored:** 2026

## 1. Import Required Libraries

In [ ]:
import os
import sys
import numpy as np
import mne

# Add the WMspatiotemporal module directory to path
sys.path.insert(0, os.path.join(os.getcwd(), 'WMspatiotemporal'))

from STWM_functions import (
    stc_merger,
    source_estimate_average_visual_checher,
    statistical_inference,
    stat_visualization,
)
from Script_for_Figure_generating import fig_5

print(f'MNE version: {mne.__version__}')
print('✓ All imports successful')

## 2. Configuration

Set all analysis parameters here before running the pipeline.

In [ ]:
import yaml

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# ── COMPUTING ─────────────────────────────────────────────────────────────────
n_jobs           = config['processing']['n_jobs']
num_subject      = config['source_group_statistics']['num_subjects']

# ── FILE NAVIGATION ───────────────────────────────────────────────────────────
folder           = config['paths']['data_folder']
subject_name     = config['subject']['subject_name']
subjects_dir     = config['paths']['subjects_dir']

# ── SOURCE SPACE ──────────────────────────────────────────────────────────────
spacing          = config['source_space']['spacing']

# ── CONDITION LABELS ──────────────────────────────────────────────────────────
condition_1      = config['conditions']['condition_1']['name']                  # spatial
condition_2      = config['conditions']['condition_2']['name']                  # temporal
condition_3      = config['source_group_statistics']['condition_baseline']      # baseline

# ── FREQUENCY BAND ────────────────────────────────────────────────────────────
freq_min         = config['source_estimate']['freq_min']
freq_max         = config['source_estimate']['freq_max']

# ── SOURCE ESTIMATE ───────────────────────────────────────────────────────────
orientation      = config['source_estimate']['orientation']

# ── VISUALIZATION OPTIONS ─────────────────────────────────────────────────────
hemi             = config['visualization']['hemi']
surface          = config['visualization']['surface']
views            = config['visualization']['views']
colormap         = config['source_visualization']['colormap']
time_label       = config['source_visualization']['time_label']
transparency     = config['source_visualization']['transparency']
time_viewer      = config['source_visualization']['time_viewer']
volume_options   = config['source_visualization']['volume_options']
view_layout      = config['source_visualization']['view_layout']
annotation       = config['source_visualization']['annotation']
mode             = config['source_visualization']['mode']
backend          = config['source_visualization']['backend']

# ── GROUP STATISTICS ──────────────────────────────────────────────────────────
p_threshold      = config['source_group_statistics']['p_threshold']
n_permutations   = config['source_group_statistics']['n_permutations']
out_type         = config['source_group_statistics']['out_type']
buffer_size      = config['source_group_statistics']['buffer_size']   # None when null in YAML
alpha_level      = config['source_group_statistics']['alpha_level']
tstep            = config['source_group_statistics']['tstep']

print('✓ Configuration loaded from config.yaml')
print(f'  Subjects     : {num_subject}')
print(f'  Frequency    : {freq_min}–{freq_max} Hz')
print(f'  Conditions   : {condition_1} vs {condition_2} (baseline: {condition_3})')
print(f'  Orientation  : {orientation}')
print(f'  Permutations : {n_permutations}  |  p-threshold: {p_threshold}  |  α: {alpha_level}')

## 3. Prerequisites Check

Verify that the morphed STC files and the fsaverage source space produced by
`example_usage_source_complete.ipynb` are present before running the group pipeline.

In [ ]:
import os.path as op

missing = []

# Check fsaverage source space
av_src = op.join(folder, 'Av-{}-src.fif'.format(spacing))
if not op.exists(av_src):
    missing.append(av_src)

# Check individual morphed STC files for all subjects and conditions
for i in range(1, num_subject + 1):
    for cond in [condition_1, condition_2, condition_3]:
        stc_file = op.join(
            folder,
            'S{}_from_{}_to_{}_{}_morph_surf_{}_stc-lh.stc'.format(
                i, freq_min, freq_max, orientation, cond
            )
        )
        if not op.exists(stc_file):
            missing.append(stc_file)

if missing:
    print('✗ Missing files:')
    for f in missing:
        print(f'  {f}')
    print('\nRun example_usage_source_complete.ipynb first.')
else:
    print('✓ All prerequisite files found — ready to proceed')

## 4. Step 1: Load and Merge Source Estimates

Load the morphed source estimates for all subjects and all three conditions
(`condition_1`, `condition_2`, `condition_3`) and collect them into lists
ready for group-level analysis.

In [ ]:
print('Loading and merging morphed source estimates...')
print(f'  Subjects: 1 – {num_subject}')
print(f'  Band    : {freq_min}–{freq_max} Hz  |  orientation: {orientation}')
print()

stc_s, stc_t, stc_a = stc_merger(
    folder, num_subject,
    subject_name, freq_min, freq_max,
    orientation, condition_1, condition_2, condition_3,
    subjects_dir, spacing
)

print('\n' + '='*60)
print('Merge Summary')
print('='*60)
print(f'  Condition "{condition_1}" (spatial)  : {len(stc_s)} subjects loaded')
print(f'  Condition "{condition_2}" (temporal) : {len(stc_t)} subjects loaded')
print(f'  Condition "{condition_3}" (baseline) : {len(stc_a)} subjects loaded')
print(f'  Source space shape : {stc_s[0].data.shape}  (vertices × time/freq)')
print('\n✓ Merge complete')

## 5. Step 2: Visual Check of Group Average

Inspect the normalised contrast `(S − T) / baseline` for one subject on the
`fsaverage` brain to verify that the morphed estimates look reasonable before
running the computationally expensive permutation test.

In [ ]:
subject_to_visualize = 0   # 0-based index into stc_s / stc_t / stc_a lists

print(f'Visualising subject index {subject_to_visualize} ...')

brain_check = source_estimate_average_visual_checher(
    stc_s, stc_t, stc_a,
    subject_to_visualize, freq_min, freq_max, spacing,
    hemi, colormap, time_label,
    transparency,        # passed as the `alpha` argument of the function
    time_viewer,
    views, volume_options, view_layout, surface,
    annotation, mode, subjects_dir, backend
)

print(f'✓ Visual check complete — image saved as '
      f'"{subject_to_visualize}_from_{freq_min}_to_{freq_max}.png"')

## 6. Step 3: Group-Level Statistics

Run a **spatio-temporal cluster-based one-sample permutation test** on the
normalised contrast `(condition_2 − condition_1) / baseline` across all subjects.
This test accounts for the spatial correlation between neighbouring sources.

> **Note:** With `n_permutations = 2000` this step may take several minutes
> depending on `n_jobs` and the number of sources.

In [ ]:
print('Running spatio-temporal cluster permutation test...')
print(f'  Subjects       : {num_subject}')
print(f'  Permutations   : {n_permutations}')
print(f'  p-threshold    : {p_threshold}  (cluster-forming)')
print(f'  α-level        : {alpha_level}  (significance)')
print(f'  n_jobs         : {n_jobs}')
print()

stc_all_cluster_vis, stc_new, clu = statistical_inference(
    num_subject, stc_s, stc_t, stc_a,
    spacing, folder, subjects_dir,
    p_threshold, n_permutations, tstep,
    n_jobs, out_type, buffer_size, alpha_level
)

T_obs, clusters, cluster_p_values, H0 = clu

print('\n' + '='*60)
print('Statistics Summary')
print('='*60)
print(f'  T-statistic map shape : {T_obs.shape}')
print(f'  Total clusters found  : {len(clusters)}')
print(f'  Significant clusters  : {np.sum(cluster_p_values < alpha_level)}')

if np.any(cluster_p_values < alpha_level):
    print(f'\n  Significant clusters (p < {alpha_level}):')
    for idx, p in enumerate(cluster_p_values):
        if p < alpha_level:
            n_verts = clusters[idx][0].shape[0] if len(clusters[idx]) > 0 else 0
            print(f'    Cluster {idx:2d}: p = {p:.4f}  |  {n_verts} vertices')
else:
    print(f'\n  No significant clusters at α = {alpha_level}')

print('\n✓ Statistics complete')

## 7. Step 4: Visualize Statistical Results

Project the significant cluster map onto the `fsaverage` inflated brain.
The plotted values are the cluster T-statistics weighted by the mean contrast,
so the sign reflects which condition dominates.

In [ ]:
print('Rendering statistical map on the cortical surface...')

brain = stat_visualization(
    stc_new, freq_min, freq_max, spacing,
    hemi, colormap, time_label, transparency, time_viewer,
    views, volume_options, view_layout, surface,
    annotation, mode, subjects_dir, backend
)

print(f'✓ Visualization complete — image saved as '
      f'"Average_statistics_from_{freq_min}_to_{freq_max}.png"')

## 8. Step 5 (Optional): Figure for Publication

Compose a publication-ready two-panel figure comparing the statistical maps for
two frequency bands side by side.  
Adjust `freq_min_1 / freq_max_1` and `freq_min_2 / freq_max_2` to the bands
you want to compare, and set `vmin` / `vmax` to the colour-scale limits
(use the *"Using control points"* output of the brain viewer as a guide).

In [ ]:
import matplotlib.pyplot as plt

# ── FIGURE PARAMETERS ─────────────────────────────────────────────────────────
freq_min_1 = 31
freq_max_1 = 80
freq_min_2 = 31
freq_max_2 = 80
vmin       = 0.32   # colorbar minimum (adjust after inspecting brain viewer output)
vmax       = 0.57   # colorbar maximum

# ── GENERATE FIGURE ───────────────────────────────────────────────────────────
print(f'Generating publication figure ...')
print(f'  Panel 1: {freq_min_1}–{freq_max_1} Hz')
print(f'  Panel 2: {freq_min_2}–{freq_max_2} Hz')
print(f'  Colour scale: [{vmin}, {vmax}]')

fig = fig_5(freq_min_1, freq_max_1, freq_min_2, freq_max_2, vmin, vmax)

plt.savefig(
    'Figure_source_statistics_{}_{}_vs_{}_{}.png'.format(
        freq_min_1, freq_max_1, freq_min_2, freq_max_2
    ),
    dpi=300, bbox_inches='tight'
)
plt.show()
print('✓ Figure saved')

## Notes

- **Prerequisites:** Run `example_usage_source_complete.ipynb` for every subject
  before executing this notebook.
- **Path handling:** Make sure `folder` and `subjects_dir` point to the same
  directories used during individual analysis so that file names match.
- **Orientation:** `orientation` must be identical to the value used when saving
  the morphed STC files (e.g., `'fix'`).
- **Step 3 runtime:** The permutation test scales with `num_subject × n_permutations × n_sources`.
  Increase `n_jobs` or reduce `n_permutations` to speed up exploration.
- **Visual check index:** `subject_to_visualize` is 0-based (0 = first subject).
- **Publication figure (Step 5):** Requires the PNG images produced in Step 4
  to be present in the working directory.
  Set `vmin` / `vmax` using the *'Using control points'* values printed by
  the brain viewer during visualisation.